##### Imports and Install


In [3]:
import torch
import torch.nn as nn
from torchvision import models



In [ ]:
#pip install torch torchvision

##### Recreate Architecture

In [ ]:
NUM_CLASSES = 4
device      = torch.device('cpu')  # Conversion must be done on CPU

mobilenetv2 = models.mobilenet_v2(weights=None)

for param in mobilenetv2.parameters():
    param.requires_grad = False

for param in mobilenetv2.features[-3:].parameters():
    param.requires_grad = True

in_features                = mobilenetv2.classifier[1].in_features
mobilenetv2.classifier[1]  = nn.Linear(in_features, NUM_CLASSES)

mobilenetv2.load_state_dict(
    torch.load('I:/Capstone/Model/mobilenetv2_trained.pth', map_location=device)
)

mobilenetv2.eval()
print("Model loaded successfully")

##### Convert to PyTorch Mobile format

In [ ]:
from torch.utils.mobile_optimizer import optimize_for_mobile

# Step 1 — Trace the model
# This records the model's operations on a dummy input
# so it can be run without Python on Android
dummy_input   = torch.randn(1, 3, 224, 224)
traced_model  = torch.jit.trace(mobilenetv2, dummy_input)

# Step 2 — Optimise for mobile
# Strips out anything not needed for inference
# reduces file size and improves performance
optimised_model = optimize_for_mobile(traced_model)

# Step 3 — Save
ptl_path = 'I:/Capstone/Model/mobilenetv2.ptl'
optimised_model._save_for_lite_interpreter(ptl_path)

print(f"PyTorch Mobile model saved to : {ptl_path}")

import os
size_mb = os.path.getsize(ptl_path) / (1024 * 1024)
print(f"File size                     : {size_mb:.2f} MB")

##### Verify Model Works

In [ ]:
# Load the saved model back and run a test inference
# to confirm it works correctly before moving to Android
from torch.jit import load

loaded_model = load(ptl_path)
loaded_model.eval()

test_input  = torch.randn(1, 3, 224, 224)
test_output = loaded_model(test_input)

print(f"Output shape : {test_output.shape}")  # Should be [1, 4]
print(f"Raw outputs  : {test_output}")
print(f"Predicted class index : {test_output.argmax(dim=1).item()}")
print("Model verified successfully")